# LoRA on a small layer and quantization memory estimates

Low-rank adaptation freezes a pretrained matrix (W) and learns ΔW = BA with rank (r). This notebook verifies the algebra, trains only adapter parameters in a toy layer and estimates how quantization changes parameter memory. It does not train a quantized language model.

In [2]:
import numpy as np
import pandas as pd
from nlp_portfolio.lora import lora_parameter_count, lora_linear, model_memory_gib

in_features, out_features, rank = 4096, 4096, 16
full = in_features * out_features
adapter = lora_parameter_count(in_features, out_features, rank)
print(f"full matrix parameters: {full:,}")
print(f"LoRA parameters: {adapter:,} ({100 * adapter / full:.3f}% of full matrix)")

full matrix parameters: 16,777,216
LoRA parameters: 131,072 (0.781% of full matrix)


In [3]:
parameter_counts = [1_000_000_000, 7_000_000_000, 13_000_000_000]
memory = []
for count in parameter_counts:
    for bits in [32, 16, 8, 4]:
        memory.append({"parameters": f"{count/1e9:g}B", "precision_bits": bits,
                       "weight_memory_GiB": model_memory_gib(count, bits)})
pd.DataFrame(memory).pivot(index="parameters", columns="precision_bits",
                           values="weight_memory_GiB").round(2)

precision_bits,4,8,16,32
parameters,,,,
13B,6.05,12.11,24.21,48.43
1B,0.47,0.93,1.86,3.73
7B,3.26,6.52,13.04,26.08


The table covers weights only. Training also stores activations, gradients, optimizer states and temporary kernels. QLoRA keeps the frozen backbone quantized while adapter computation and optimizer states use higher precision.

In [5]:
rng = np.random.default_rng(42)
x = rng.normal(size=(5, 8))
weight = rng.normal(size=(6, 8))
a = rng.normal(size=(8, 2))
b = rng.normal(size=(2, 6))
alpha = 4

factorized = lora_linear(x, weight, a, b, alpha=alpha)
merged_weight = weight + (alpha / a.shape[1]) * (a @ b).T
merged = x @ merged_weight.T
print("maximum factorized-versus-merged difference:", np.max(np.abs(factorized - merged)))
assert np.allclose(factorized, merged)

maximum factorized-versus-merged difference: 1.7763568394002505e-15


## Trainable adapter layer

The base weight is frozen. Matrix `B` starts at zero, so the network initially reproduces the base model exactly. Gradients then learn a task-specific low-rank update.

In [7]:
import torch
from torch import nn

class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=8):
        super().__init__()
        self.base = base
        self.base.weight.requires_grad_(False)
        if self.base.bias is not None:
            self.base.bias.requires_grad_(False)
        self.a = nn.Parameter(torch.randn(base.in_features, rank) * 0.01)
        self.b = nn.Parameter(torch.zeros(rank, base.out_features))
        self.scale = alpha / rank

    def forward(self, inputs):
        return self.base(inputs) + self.scale * inputs @ self.a @ self.b

torch.manual_seed(42)
layer = LoRALinear(nn.Linear(12, 3), rank=2, alpha=4)
features = torch.randn(64, 12)
targets = torch.randint(0, 3, (64,))
optimizer = torch.optim.AdamW([layer.a, layer.b], lr=0.05)
for _ in range(80):
    optimizer.zero_grad()
    loss = nn.CrossEntropyLoss()(layer(features), targets)
    loss.backward()
    optimizer.step()
print("final toy loss:", round(loss.item(), 4))
print("trainable parameters:", sum(p.numel() for p in layer.parameters() if p.requires_grad))

final toy loss: 0.8694
trainable parameters: 30


## Optional Hugging Face PEFT configuration

Target-module names depend on architecture. They should be inspected from `model.named_modules()` instead of copied blindly.

In [9]:
RUN_HF_PEFT = False
if RUN_HF_PEFT:
    from transformers import AutoModelForSequenceClassification
    from peft import LoraConfig, get_peft_model
    base = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
    config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                        target_modules=["q_lin", "v_lin"], task_type="SEQ_CLS")
    adapted = get_peft_model(base, config)
    adapted.print_trainable_parameters()

Rank, target layers and learning rate are validation hyperparameters. Memory estimates must be measured on the target hardware because kernel choice, sequence length and activation checkpointing can dominate the static parameter calculation.